# MESA transfer learning for sleep stage classification

This notebook pretrains a Random Forest classifier on MESA Sleep dataset features
then saves the model for fine-tuning on personal overnight data.

MESA provides actigraphy and PSG annotations at 30-second epochs for 2,056 participants.
The 30-second epoch size matches the window size used throughout the Basecamp pipeline.

**Data access:** MESA data is available from the National Sleep Research Resource (NSRR)
under a Data Use Agreement. Raw data files must not be committed to this repository.
See docs/adr/ADR-010-mesa-transfer-learning.md for the full DAUA terms.

**Required acknowledgement:** NSRR R24 HL114473: NHLBI National Sleep Research Resource.

When real MESA files are not present, this notebook generates synthetic data with the
same column structure and runs end-to-end so the pipeline can be reviewed without data access.

## Section 1: Data loading

Set `ACTIGRAPHY_PATH` and `PSG_PATH` to your local MESA file paths.
If either path is missing, synthetic stand-in data is generated instead.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # headless backend for CI environments
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# ── File paths ────────────────────────────────────────────────────────────────
# Set these to your downloaded MESA file paths before running with real data.
ACTIGRAPHY_PATH = "../data/mesa/mesa-sleep-actigraphy.csv"
PSG_PATH        = "../data/mesa/mesa-sleep-psg-staging.csv"
MODEL_OUT       = "../data/mesa_pretrained.pkl"
FIGURE_OUT      = "../data/mesa_feature_importance.png"

N_SYNTHETIC_PARTICIPANTS = 80
EPOCHS_PER_PARTICIPANT   = 480   # 4 hours at 30-second epochs

USING_SYNTHETIC = not (os.path.exists(ACTIGRAPHY_PATH) and os.path.exists(PSG_PATH))

if USING_SYNTHETIC:
    print("MESA files not found. Generating synthetic stand-in data.")
    print(f"  Expected actigraphy: {ACTIGRAPHY_PATH}")
    print(f"  Expected PSG:        {PSG_PATH}")
    print()
else:
    print("MESA files found. Loading real data.")

In [ ]:
def _make_synthetic_actigraphy(n_participants: int, epochs_per: int) -> pd.DataFrame:
    """Generate synthetic actigraphy shaped like MESA output."""
    rng = np.random.default_rng(42)
    rows = []
    for pid in range(n_participants):
        for epoch in range(epochs_per):
            # Activity count: low during sleep, high during wake
            # Simulate a rough sleep architecture: wake at start, deep in middle, REM at end
            relative_pos = epoch / epochs_per
            if relative_pos < 0.1 or relative_pos > 0.9:
                base_activity = rng.exponential(120)
            elif 0.2 < relative_pos < 0.5:
                base_activity = rng.exponential(5)   # deep sleep
            else:
                base_activity = rng.exponential(30)  # light / REM
            rows.append({
                "participant_id":     pid,
                "epoch_time":         epoch * 30,
                "activity_count":     float(max(0, base_activity)),
                "zero_crossing_rate": float(max(0, rng.normal(0.3 if base_activity < 20 else 0.7, 0.1))),
            })
    return pd.DataFrame(rows)


def _make_synthetic_psg(actigraphy_df: pd.DataFrame) -> pd.DataFrame:
    """Generate synthetic PSG labels that loosely match the actigraphy signal."""
    rng = np.random.default_rng(42)
    rows = []
    for pid in actigraphy_df["participant_id"].unique():
        sub = actigraphy_df[actigraphy_df["participant_id"] == pid].copy()
        for _, row in sub.iterrows():
            ac = row["activity_count"]
            if ac > 80:
                stage = 0    # Wake
            elif ac < 8:
                stage = 3    # N3 (deep)
            elif row["zero_crossing_rate"] > 0.6 and ac < 40:
                stage = 5    # REM
            elif ac < 30:
                stage = 2    # N2
            else:
                stage = 1    # N1
            # Add label noise
            if rng.random() < 0.05:
                stage = rng.choice([0, 1, 2, 3, 5])
            rows.append({"participant_id": pid, "epoch_time": row["epoch_time"], "stage": stage})
    return pd.DataFrame(rows)


if USING_SYNTHETIC:
    actigraphy_df = _make_synthetic_actigraphy(N_SYNTHETIC_PARTICIPANTS, EPOCHS_PER_PARTICIPANT)
    psg_df        = _make_synthetic_psg(actigraphy_df)
else:
    actigraphy_df = pd.read_csv(ACTIGRAPHY_PATH)
    psg_df        = pd.read_csv(PSG_PATH)

print(f"Actigraphy rows:  {len(actigraphy_df):,}")
print(f"PSG rows:         {len(psg_df):,}")
print(f"Participants:     {actigraphy_df['participant_id'].nunique()}")
print()
print("Actigraphy columns:", list(actigraphy_df.columns))
print("PSG columns:",        list(psg_df.columns))

## Section 2: Feature engineering

Eight features are computed per epoch. These are chosen for their overlap with
features the Basecamp pipeline extracts from CSI and radar.

PSG stages are mapped to three classes:
- 0 (Wake) -> WAKE
- 1 (N1), 2 (N2), 5 (REM) -> LIGHT_REM
- 3 (N3) -> DEEP

In [ ]:
# ── Merge actigraphy and PSG on participant + epoch ───────────────────────────
merged = pd.merge(
    actigraphy_df,
    psg_df[["participant_id", "epoch_time", "stage"]],
    on=["participant_id", "epoch_time"],
    how="inner",
)

# ── Three-class label mapping ─────────────────────────────────────────────────
# PSG stage codes: 0=Wake, 1=N1, 2=N2, 3=N3, 5=REM
STAGE_MAP = {0: 0, 1: 2, 2: 2, 3: 1, 5: 2}  # Wake=0, Deep=1, Light/REM=2
CLASS_NAMES = ["Wake", "Deep", "Light/REM"]

merged["label"] = merged["stage"].map(STAGE_MAP)
merged = merged.dropna(subset=["label"])
merged["label"] = merged["label"].astype(int)

print("Class distribution:")
for cls_id, cls_name in enumerate(CLASS_NAMES):
    n = (merged["label"] == cls_id).sum()
    pct = 100 * n / len(merged)
    print(f"  {cls_name:12s}  {n:7,}  ({pct:.1f}%)")

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute 8 features per epoch. Operates per-participant to avoid cross-person leakage."""
    results = []
    for pid, group in df.groupby("participant_id", sort=False):
        g = group.sort_values("epoch_time").copy()
        n = len(g)

        # Time-within-night features
        g["epoch_position"]        = np.arange(n) / max(n - 1, 1)
        g["time_since_sleep_onset"] = (g["epoch_time"] - g["epoch_time"].iloc[0]) / 3600.0

        # Rolling activity statistics (5-epoch window)
        g["rolling_mean_activity_5"] = (
            g["activity_count"].rolling(5, min_periods=1, center=True).mean()
        )
        g["rolling_std_activity_5"] = (
            g["activity_count"].rolling(5, min_periods=1, center=True).std().fillna(0)
        )

        # Stage transition features (require label column; present during training)
        if "label" in g.columns:
            prev_label = g["label"].shift(1).fillna(g["label"].iloc[0])
            g["consecutive_same_stage"] = (
                (g["label"] == prev_label)
                .groupby((g["label"] != prev_label).cumsum())
                .cumcount() + 1
            ).values
            g["previous_stage"] = prev_label.values
        else:
            g["consecutive_same_stage"] = 1
            g["previous_stage"]         = 2  # default to Light/REM

        results.append(g)

    return pd.concat(results, ignore_index=True)


FEATURE_COLS = [
    "activity_count",
    "zero_crossing_rate",
    "epoch_position",
    "time_since_sleep_onset",
    "rolling_mean_activity_5",
    "rolling_std_activity_5",
    "consecutive_same_stage",
    "previous_stage",
]

featured = engineer_features(merged)
print(f"Feature matrix shape: {featured[FEATURE_COLS].shape}")
featured[FEATURE_COLS].describe().round(3)

## Section 3: Pretraining

The train/test split is by participant, not by epoch, to prevent data leakage.
Participants in the test set have no epochs in the training set.

In [ ]:
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ── Participant-level train / test split ──────────────────────────────────────
rng = np.random.default_rng(0)
all_pids  = np.array(featured["participant_id"].unique())
rng.shuffle(all_pids)
split     = int(0.8 * len(all_pids))
train_pids = set(all_pids[:split])
test_pids  = set(all_pids[split:])

train_mask = featured["participant_id"].isin(train_pids)
test_mask  = featured["participant_id"].isin(test_pids)

X_train = featured.loc[train_mask, FEATURE_COLS].values
y_train = featured.loc[train_mask, "label"].values
X_test  = featured.loc[test_mask,  FEATURE_COLS].values
y_test  = featured.loc[test_mask,  "label"].values

print(f"Train: {len(X_train):,} epochs from {len(train_pids)} participants")
print(f"Test:  {len(X_test):,} epochs from {len(test_pids)} participants")

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Overall accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalised)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.6 else "black")
fig.tight_layout()
plt.show()

# ── Save model ────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
with open(MODEL_OUT, "wb") as f:
    pickle.dump({"model": clf, "feature_cols": FEATURE_COLS, "class_names": CLASS_NAMES}, f)
print(f"Pretrained model saved: {MODEL_OUT}")

## Section 4: Feature importances

The plot below shows which features the pretrained model relies on most.
Features that are important here are the ones most worth preserving
or improving in the CSI feature extraction layer.

In [ ]:
importances = clf.feature_importances_
sorted_idx  = np.argsort(importances)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    [FEATURE_COLS[i] for i in sorted_idx],
    importances[sorted_idx],
    color="steelblue",
)
ax.set_xlabel("Mean decrease in impurity")
ax.set_title("Feature importances (MESA pretrained model)")
fig.tight_layout()
plt.savefig(FIGURE_OUT, dpi=120)
plt.show()
print(f"Figure saved: {FIGURE_OUT}")

top5 = sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 features:")
for name, score in top5:
    print(f"  {name:30s}  {score:.4f}")

**Interpretation.**

`activity_count` and `rolling_mean_activity_5` are expected to rank highest because
movement is the clearest discriminator between wake and sleep. `consecutive_same_stage`
captures the temporal stability of each stage, which is a strong prior for deep sleep
(long uninterrupted runs) versus REM (shorter blocks with intervening light sleep).
`zero_crossing_rate` captures breathing regularity, which is the main discriminator
between deep sleep and the light/REM class once movement is low.

When fine-tuning on personal data, features with low importance in the MESA model
may still become important for a specific individual. The fine-tuned model is allowed
to reweight all features.

## Section 5: Fine-tuning on personal data (stub)

This section requires at least 30 personal nights in the Basecamp database.
It will load the pretrained model, compute matching features from CSI and radar
signals, and continue training on the personal data with a lower learning rate.

In [ ]:
import sys
sys.path.insert(0, "..")

from server.config import DB_PATH
from server.db import get_connection

with get_connection(DB_PATH) as conn:
    row = conn.execute("SELECT COUNT(*) AS n FROM sleep_sessions WHERE end_time IS NOT NULL").fetchone()
    n_sessions = row["n"]

print(f"Personal sessions in database: {n_sessions}")

if n_sessions < 30:
    print(f"Fine-tuning requires 30 sessions. {30 - n_sessions} more nights needed.")
    print("Re-run this section once enough data has been collected.")
else:
    print("Sufficient data available. Fine-tuning implementation goes here.")
    # TODO: load personal feature vectors from the database, align with pretrained
    # feature columns, and call clf.fit() on the combined MESA + personal dataset
    # with personal epochs weighted higher to bias the model toward individual physiology.